In [18]:
import pyarrow.dataset as ds
import pandas as pd
from pathlib import Path
import numpy as np
import statsmodels.api as sm
import os
from tqdm import tqdm
import scipy
import gseapy as gp
import matplotlib.pyplot as plt
import pybiomart

In [3]:
ROOT = Path("/home/hanwenying")
DATA = ROOT / "rothman-data/gltd"
OUT  = ROOT / "rothman-anna/gltd/out"
BETA = OUT / "betas"
RESID = OUT / "resid"
SUMMARY = OUT / "summary"

In [4]:
kegg_symbols = gp.get_library(name='KEGG_2026', organism='Human')

In [50]:
genesymbols = pd.read_csv(DATA / "ensembl_genesymbol_map.txt", sep='\t')

# FOR EVERY TISSUE

In [94]:
rnk = pd.read_csv(RESID / "Brain - Cerebellum.tsv", sep='\t')

In [95]:
ensembl = pd.DataFrame(rnk['GENE'].str.split('.').str[0])
ensembl.columns = ['ENSEMBL']
rnksymbols = pd.merge(ensembl, genesymbols, on='ENSEMBL').dropna(subset=['SYMBOL']).set_index('ENSEMBL')

In [96]:
rnk['GENE'] = ensembl
rnk = rnk.set_index('GENE')
rnk_gsea = rnk.join(rnksymbols).dropna(subset=['SYMBOL']).set_index('SYMBOL')

In [97]:
rnk_betas = rnk_gsea['BETA']
rnk_resid = rnk_gsea['RESIDUAL']

In [100]:
pr_betas = gp.prerank(rnk=rnk_betas,
				gene_sets='KEGG_2026',
				threads=20,
				min_size=5,
				max_size=10000,
				permutation_num=5000,
				outdir=None,
				seed=24,
				verbose=True)

pr_resid = gp.prerank(rnk=rnk_resid,
				gene_sets='KEGG_2026',
				threads=20,
				min_size=5,
				max_size=10000,
				permutation_num=5000,
				outdir=None,
				seed=24,
				verbose=True)

2026-05-04 02:55:02,240 [INFO] Input gene rankings contains duplicated IDs
2026-05-04 02:55:02,249 [INFO] Parsing data files for GSEA.............................
2026-05-04 02:55:02,303 [INFO] Enrichr library gene sets already downloaded in: /home/hanwenying/.cache/gseapy, use local file
2026-05-04 02:55:02,344 [INFO] 0002 gene_sets have been filtered out when max_size=10000 and min_size=5
2026-05-04 02:55:02,346 [INFO] 0350 gene_sets used for further statistical testing.....
2026-05-04 02:55:02,347 [INFO] Start to run GSEA...Might take a while..................
2026-05-04 02:55:19,027 [INFO] Congratulations. GSEApy runs successfully................

2026-05-04 02:55:19,218 [INFO] Input gene rankings contains duplicated IDs
2026-05-04 02:55:19,226 [INFO] Parsing data files for GSEA.............................
2026-05-04 02:55:19,281 [INFO] Enrichr library gene sets already downloaded in: /home/hanwenying/.cache/gseapy, use local file
2026-05-04 02:55:19,322 [INFO] 0002 gene_sets have

In [103]:
pr_betas.res2d.to_csv(ROOT / "pr_betas.tsv", sep='\t', index=False)
pr_resid.res2d.to_csv(ROOT / "pr_resid.tsv", sep='\t', index=False)

In [106]:
b = pr_betas.res2d
r = pr_resid.res2d

In [113]:
b

,Name,Term,ES,NES,NOM p-val,FDR q-val,FWER p-val,Tag %,Gene %,Lead_genes
0,prerank,GRAFT-VERSUS-HOST DISEASE,0.762446,3.398051,0.0,0.0,0.0,11/21,6.54%,HLA-DRB5;HLA-DRB1;HLA-DRA;HLA-DPA1;HLA-F;HLA-C...
1,prerank,COMPLEMENT AND COAGULATION CASCADES,0.547811,3.281143,0.0,0.0,0.0,25/63,6.02%,VWF;C1QB;C1QC;C1QA;VSIG4;SERPING1;C4A;SERPINA1...
2,prerank,INTESTINAL IMMUNE NETWORK FOR IGA PRODUCTION,0.661213,3.277282,0.0,0.0,0.0,12/27,6.51%,HLA-DRB5;HLA-DRB1;HLA-DRA;LTBR;TNFSF13;HLA-DPA...
3,prerank,ALLOGRAFT REJECTION,0.738935,3.27454,0.0,0.0,0.0,11/21,6.54%,HLA-DRB5;HLA-DRB1;HLA-DRA;HLA-DPA1;HLA-F;HLA-C...
4,prerank,VIRAL PROTEIN INTERACTION WITH CYTOKINE AND CY...,0.583097,3.219253,0.0,0.0,0.0,23/49,6.75%,CXCL1;CCL19;TNFRSF1A;CXCL2;ACKR3;LTBR;TNFRSF10...
...,...,...,...,...,...,...,...,...,...,...
345,prerank,TRANSCRIPTIONAL MISREGULATION IN CANCER,-0.142714,-0.55355,0.997597,0.999215,1.0,52/144,41.27%,SLC45A3;CDK14;ETV1;RXRB;ETV5;JMJD1C;FUT8;FUS;G...
346,prerank,ALPHA-LINOLENIC ACID METABOLISM,-0.183611,-0.531046,0.973257,0.999872,1.0,2/17,10.37%,PLA2G4B;PLA2G12A
347,prerank,SIGNALING PATHWAYS REGULATING PLURIPOTENCY OF ...,-0.138021,-0.525769,0.997394,0.997346,1.0,52/115,51.15%,WNT9A;PIK3R3;ID2;SLC25A23;AKT1;CLSTN1;PIK3CB;P...
348,prerank,HIPPO SIGNALING PATHWAY - MULTIPLE SPECIES,-0.167445,-0.524572,0.980455,0.994199,1.0,9/25,37.34%,RASSF2;FRMD6;WTIP;STK24;PKN1;SAV1;AJUBA;RASSF4...


In [112]:
scipy.stats.spearmanr(b['NES'], r['NES'])

SignificanceResult(statistic=np.float64(-0.19651990162018815), pvalue=np.float64(0.00021599300628153344))